# Heavy-Tailed ABMs: OFC

This notebook reproduces power-law experiments from **Olami-Feder-Christensen (OFC)** agent-based model.

Sections:
1. Setup & imports
2. OFC simulation demo
3. OFC phase diagram
4. OFC calibration (MLE + ABC)

## 1. Setup & imports

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings('ignore')

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.dirname('.'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

from models.ofc import simulate_ofc
from utils.powerlaw_fit import fit_powerlaw, gutenberg_richter_b
from utils.plotting import (plot_ccdf, FIGURES_DIR, _apply_style, _ensure_figures_dir)

_ensure_figures_dir()

## 2. OFC Simulation Demo

Single run with `alpha_ofc=0.20`, `L=64`, `n_events=20_000`, `seed=42`.

In [2]:
# OFC demo parameters (fast: <10 s)
OFC_L = 64
OFC_ALPHA = 0.20
OFC_EVENTS = 20_000

sizes_ofc = simulate_ofc(OFC_L, OFC_ALPHA, OFC_EVENTS, seed=42)

print(f"Avalanche sizes: min={sizes_ofc.min()}, max={sizes_ofc.max()}")
print(f"Mean size: {sizes_ofc.mean():.2f}")
print(f"Fraction size-1: {(sizes_ofc == 1).mean():.3f}")

# Plot 1: first 2 000 events
_apply_style()
fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.vlines(np.arange(2000), 1, sizes_ofc[:2000], linewidth=0.4, color="darkorange")
ax.set_yscale("log")
ax.set_xlabel("Event index")
ax.set_ylabel("Avalanche size")
ax.set_title(rf"OFC avalanche time series ($L={OFC_L}$, $\alpha_{{\mathrm{{OFC}}}}={OFC_ALPHA}$)")
fig.tight_layout()
save_ts_ofc = os.path.join(FIGURES_DIR, "ofc_demo_timeseries.pdf")
fig.savefig(save_ts_ofc)
plt.close(fig)
print("Saved:", save_ts_ofc)

Avalanche sizes: min=0, max=65
Mean size: 0.45
Fraction size-1: 0.080
Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_demo_timeseries.pdf


In [3]:
# Plot 2: CCDF with power-law fit overlay
fit_res_ofc = fit_powerlaw(sizes_ofc.astype(float), xmin=2, discrete=True)

_apply_style()
fig, ax = plt.subplots(figsize=(3.5, 2.8))
plot_ccdf(sizes_ofc.astype(float), label="Simulated avalanches", ax=ax,
          color="darkorange", fit_result=fit_res_ofc)
ax.set_title(rf"OFC model CCDF ($L={OFC_L}$, $\alpha_{{\mathrm{{OFC}}}}={OFC_ALPHA}$)")
fig.tight_layout()
save_ccdf_ofc = os.path.join(FIGURES_DIR, "ofc_demo_ccdf.pdf")
fig.savefig(save_ccdf_ofc)
plt.close(fig)
print("Saved:", save_ccdf_ofc)

TPL params: alpha=1.362, lambda=0.0697, 1/lambda=14.3
xmin=2.0, n_tail=1332
TPL vs lognormal             : R=+2.71, p=0.007 -> TPL wins
TPL vs exponential           : R=+6.25, p=0.000 -> TPL wins
TPL vs power_law             : R=+7.36, p=0.000 -> TPL wins
TPL vs stretched_exponential : R=+1.77, p=0.076 -> TPL wins
Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_demo_ccdf.pdf


In [4]:
# Plot 3: TPL cutoff scale 1/lambda vs alpha_ofc
import powerlaw
from scipy.integrate import quad

ALPHA_SWEEP = np.linspace(0.10, 0.25, 10)
cutoffs = []

for alpha_ofc in ALPHA_SWEEP:
    s = simulate_ofc(64, alpha_ofc, n_events=50000, seed=0)
    s = s[s >= 2].astype(float)
    fit_s = powerlaw.Fit(s, xmin=2, discrete=True, verbose=False)
    lam = fit_s.truncated_power_law.Lambda
    cutoffs.append(1 / lam)
    print(f"alpha_ofc={alpha_ofc:.3f}: 1/lambda={1/lam:.1f}")

_apply_style()
fig, ax = plt.subplots(figsize=(3.3, 2.5))
ax.plot(ALPHA_SWEEP, cutoffs, 'o-', markersize=3, linewidth=1, color='steelblue')
ax.axhline(64**2, linestyle='--', color='gray', linewidth=0.8, label=r'$L^2=4096$')
ax.set_yscale('log')
ax.set_xlabel(r'$\alpha_{\mathrm{OFC}}$')
ax.set_ylabel(r'$1/\lambda$ (blocks)')
ax.legend(frameon=False)
fig.tight_layout()
save_cutoff = os.path.join(FIGURES_DIR, "ofc_tpl_cutoff.pdf")
fig.savefig(save_cutoff)
plt.close(fig)
print("Saved:", save_cutoff)

alpha_ofc=0.100: 1/lambda=1.6
alpha_ofc=0.117: 1/lambda=1.9
alpha_ofc=0.133: 1/lambda=2.0
alpha_ofc=0.150: 1/lambda=3.0
alpha_ofc=0.167: 1/lambda=5.0
alpha_ofc=0.183: 1/lambda=7.2
alpha_ofc=0.200: 1/lambda=13.2
alpha_ofc=0.217: 1/lambda=36.5
alpha_ofc=0.233: 1/lambda=265.0
alpha_ofc=0.250: 1/lambda=13524.5
Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_tpl_cutoff.pdf


## 3. OFC Phase Diagram

Grid search over `alpha_ofc ∈ [0.10, 0.24]` and `L ∈ {32, 64, 128}`.  
Parameters: `N_EVENTS=50_000`, `N_SEEDS=10`.

In [2]:
from experiments.ofc_phase_diagram import run_ofc_phase_diagram, plot_ofc_phase_diagram

b_means, b_stds = run_ofc_phase_diagram()
plot_ofc_phase_diagram(b_means, b_stds)

OFC phase diagram: 100%|██████████| 30000000/30000000 [07:14<00:00, 68984.84ev/s] 


Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_phase_diagram.pdf


'/Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_phase_diagram.pdf'

## 4. OFC Calibration on USGS Catalog

### 4a. MLE calibration

Find `alpha_ofc*` that minimises `|b_sim − b_emp|` where `b_emp` is the
Gutenberg-Richter b-value estimated from the USGS global catalog (M ≥ 4.5,
2000–2024).

In [ ]:
from experiments.ofc_calibration import run_mle_calibration
from data.download_data import download_usgs_catalog
from utils.powerlaw_fit import gutenberg_richter_b
from experiments.ofc_calibration import run_mle_calibration, _ensure_figures_dir

_ensure_figures_dir()
catalog = download_usgs_catalog(min_magnitude=2.0, start="2000-01-01", end="2024-01-01")
magnitudes = catalog["magnitude"].dropna().values
magnitudes = magnitudes[magnitudes >= 4.5]
b_emp = gutenberg_richter_b(magnitudes, m_min=4.5)
print(f"Empirical b = {b_emp:.4f}")  
run_mle_calibration(b_emp)

Empirical b = 1.3094


OFC MLE calibration: 100%|██████████| 30000000/30000000 [12:06<00:00, 41290.67ev/s] 


  alpha_ofc* = 0.2345  |  Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_calibration_mle.pdf


(np.float64(0.23448275862068965),
 array([3.23002503, 3.05016527, 2.87375129, 2.70576705, 2.56705174,
        2.46190571, 2.34912536, 2.25244671, 2.16759215, 2.08406063,
        2.01271375, 1.94509391, 1.88618898, 1.83197825, 1.78608393,
        1.73051072, 1.70244426, 1.66680839, 1.63894672, 1.6094566 ,
        1.59200248, 1.6054748 , 1.585866  , 1.60310653, 1.6160409 ,
        1.56243057, 1.31484988, 0.74007284, 0.52062773, 0.37295332]))

### 4b. ABC calibration (skipped)

ABC-SMC calibration is **not run here** due to compute cost (hours).
To run it, execute the cell below (commented out by default).
This produces `figures/ofc_calibration_abc.pdf` with the posterior
distribution over `alpha_ofc`.

In [ ]:
# from data.download_data import download_usgs_catalog
# from utils.powerlaw_fit import gutenberg_richter_b
# from experiments.ofc_calibration import run_abc_calibration, _ensure_figures_dir

# _ensure_figures_dir()
# catalog = download_usgs_catalog(min_magnitude=2.0, start="2000-01-01", end="2024-01-01")
# magnitudes = catalog["magnitude"].dropna().values
# magnitudes = magnitudes[magnitudes >= 4.5]
# b_emp = gutenberg_richter_b(magnitudes, m_min=4.5)
# print(f"Empirical b = {b_emp:.4f}")
# run_abc_calibration(b_emp)